In [2]:
from datasets import load_dataset

dataset = load_dataset("knkarthick/samsum")

In [3]:
from transformers import AutoTokenizer

model_id = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

In [4]:
import numpy as np
from datasets import concatenate_datasets

tokenized_inputs = concatenate_datasets([dataset["train"],
           dataset["test"]]).map(lambda x: tokenizer(x["dialogue"], truncation=True),
           batched=True,
           remove_columns=["dialogue", "summary"])
input_lenghts = [len(x) for x in tokenized_inputs["input_ids"]]
max_source_length = int(np.percentile(input_lenghts, 85))

tokenized_targets = concatenate_datasets([dataset["train"],
            dataset["test"]]).map(lambda x: tokenizer(x["summary"], truncation=True),
            batched=True,
            remove_columns=["dialogue", "summary"])
target_lenghts = [len(x) for x in tokenized_targets["input_ids"]]
max_target_length = int(np.percentile(target_lenghts, 90))

Map:   0%|          | 0/15550 [00:00<?, ? examples/s]

Map:   0%|          | 0/15550 [00:00<?, ? examples/s]

In [5]:
def preprocess_function(sample,padding="max_length"):
    inputs = ["summarize: " + item for item in sample["dialogue"]]

    model_inputs = tokenizer(
        inputs,
        max_length=max_source_length,
        padding=padding,
        truncation=True
    )

    labels = tokenizer(
        text_target=sample["summary"],
        max_length=max_target_length,
        padding=padding,
        truncation=True
    )

    if padding == "max_length":
        labels["input_ids"] = [
            [(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]
        ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess_function,
                batched=True,
                remove_columns=["dialogue", "summary", "id"])

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

In [6]:
tokenized_dataset["train"].save_to_disk("data/train")
tokenized_dataset["test"].save_to_disk("data/eval")

Saving the dataset (0/1 shards):   0%|          | 0/14731 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/819 [00:00<?, ? examples/s]

In [9]:
from transformers import AutoModelForSeq2SeqLM
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [12]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
 r=16,
 lora_alpha=32,
 target_modules=["q", "v"],
 lora_dropout=0.05,
 bias="none",
 task_type=TaskType.SEQ_2_SEQ_LM
)

In [13]:
model = get_peft_model(model, lora_config)

In [15]:
from transformers import DataCollatorForSeq2Seq

label_pad_token_id = -100
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=label_pad_token_id,
    pad_to_multiple_of=8
)

In [20]:
from transformers import Seq2SeqTrainingArguments

output_dir="lora-flan-t5-small"

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    auto_find_batch_size=True,
    learning_rate=1e-3,
    num_train_epochs=3,
    logging_strategy="steps",
    logging_steps=500,
    save_strategy="no",
    report_to="none"
)

In [21]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"],
)

In [22]:
trainer.train()

/Users/joewilkinson/Projects/scratchllm/env-llm/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
500,1.930931
1000,1.907964
1500,1.870978
2000,1.855944
2500,1.841714
3000,1.827880
3500,1.830122
4000,1.816931
4500,1.779199
5000,1.820916


TrainOutput(global_step=5526, training_loss=1.8407334635924741, metrics={'train_runtime': 1760.837, 'train_samples_per_second': 25.098, 'train_steps_per_second': 3.138, 'total_flos': 4154239974113280.0, 'train_loss': 1.8407334635924741, 'epoch': 3.0})

In [23]:
peft_model_id="results"
trainer.model.save_pretrained(peft_model_id)
tokenizer.save_pretrained(peft_model_id)

('results/tokenizer_config.json', 'results/tokenizer.json')

In [26]:
import torch
from peft import PeftModel, PeftConfig

peft_model_id = "results"
config = PeftConfig.from_pretrained(peft_model_id)


base_model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
model = PeftModel.from_pretrained(base_model, peft_model_id)
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)

model = PeftModel.from_pretrained(model, peft_model_id, device_map={"":0})
model.eval()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
/Users/joewilkinson/Projects/scratchllm/env-llm/lib/python3.13/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/Users/joewilkinson/Projects/scratchllm/env-llm/lib/python3.13/site-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.encoder.block.0.layer.0.SelfAttention.q.lora_A.default.weight', 'base_model.model.base_model.model.encoder.block.0.layer.0.SelfAttention.q.lora_B.default.weight', 'base_model.model.base_model.model.encoder.block.0

PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): PeftModelForSeq2SeqLM(
      (base_model): LoraModel(
        (model): T5ForConditionalGeneration(
          (shared): Embedding(32128, 512)
          (encoder): T5Stack(
            (embed_tokens): Embedding(32128, 512)
            (block): ModuleList(
              (0): T5Block(
                (layer): ModuleList(
                  (0): T5LayerSelfAttention(
                    (SelfAttention): T5Attention(
                      (q): lora.Linear(
                        (base_layer): Linear(in_features=512, out_features=384, bias=False)
                        (lora_dropout): ModuleDict(
                          (default): Dropout(p=0.05, inplace=False)
                        )
                        (lora_A): ModuleDict(
                          (default): Linear(in_features=512, out_features=16, bias=False)
                        )
                        (lora_B): ModuleDict(
                          (default): 

In [30]:
from random import randrange

dataset = load_dataset("knkarthick/samsum")
sample = dataset['test'][randrange(len(dataset["test"]))]

input_ids = tokenizer(
    sample["dialogue"],
    return_tensors="pt",
    truncation=True
).input_ids.to('cpu')
outputs = model.generate(input_ids=input_ids, max_new_tokens=10, do_sample=True, top_p=0.9)
print(f"input sentence: {sample['dialogue']}\n{'---'* 20}")

print(f"summary:\n{tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)[0]}")

input sentence: Ollie: Okay, Kelly! Ur up nxt!
Kelly: Me? I don't wanna.
Mickey: C'mon!
Jessica: Yeah! What's yours?
Kelly: Fine. It's a sculpture garden in Finnland.
Ollie: What's scary about sculptures? Wait! Do they resemble vampires and stuff?
Mickey: Nah, I'm sure they look rly nice.
Kelly: It's not the sculptures, it's the amount of them and their faces!
Jessica: Faces? What faces?
Kelly: Well, they resemble ppl in different activities like hugging, training, doing sport and so on. But the faces are just morbid and there's like a hundred of them. All staring at you!
Ollie: Another one?
Mickey: Certainly!
Jessica: Well, Ollie, ur turn!
Ollie: Nagoro village in Japan!
Mickey: Y?
Ollie: Well, maybe it's not scary, but it similar to Kelly's place. It's just creepy as hell.
Jessica: Bt y?
Ollie: Imagine a village with ppl living in it. And in the same village u have these human-sized figures. And there's more of them than the ppl that actually live there!
Kelly: Creepy AH!
Mickey: WTF